In [1]:
import pandas as pd 
import numpy as np 


In [2]:
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

print("Data loaded successfully!")

In [3]:
df = pd.merge(orders, items, on='order_id', how='inner')
df = pd.merge(df, products, on='product_id', how='inner')
df = pd.merge(df, customers, on='customer_id', how='inner')

print(df.head(3))

In [4]:
df = df[df['order_status'] == 'delivered']
df = df.dropna()
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['purchase_week'] = df['order_purchase_timestamp'].dt.to_period('W')

print("Data cleaned! Current shape:", df.shape)

In [5]:

summary = df.groupby(['customer_state', 'product_category_name']).agg({
    'price': 'sum',          
    'order_item_id': 'count' 
}).reset_index() 


summary = summary.rename(columns={
    'price': 'total_revenue',
    'order_item_id': 'items_sold'
})

summary = summary.sort_values(by='total_revenue', ascending=False)
print(summary.head(10))

In [8]:
# 1. Load the translation file (this should be in your downloaded Kaggle folder)
translations = pd.read_csv('product_category_name_translation.csv')

# 2. Merge the English translations into your summary table
# We use a 'left' join so we don't lose any rows if a translation is missing
final_report = pd.merge(summary, translations, on='product_category_name', how='left')

# 3. Reorder the columns so the English name sits right next to the state
final_report = final_report[['customer_state', 'product_category_name_english', 'total_revenue', 'items_sold']]

# 4. Save your hard work to a brand new CSV file!
# index=False prevents pandas from exporting those random row numbers (1265, 1263, etc.)
final_report.to_csv('brazil_ecommerce_summary.csv', index=False)

# Let's peek at the final, translated English version
print("Report successfully translated and saved!")
print(final_report.head(5))

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Grab just the top 5 categories from your final report
top_5 = final_report.head(5)

# 2. Set up the visual styling (Dark mode, terminal font)
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'

# 3. Create the canvas (10 inches wide, 6 inches tall)
plt.figure(figsize=(10, 6))

# 4. Build the bar chart
# We use a horizontal bar chart (y=category, x=revenue) because category names are long
ax = sns.barplot(
    x='total_revenue', 
    y='product_category_name_english', 
    data=top_5,
    color='#00ff41'  # A sharp, matrix-style neon green
)

# 5. Add a subtle grid layout to make the values easy to read
ax.grid(color='#333333', linestyle='--', linewidth=0.5, axis='x')

# 6. Add labels and a clean title
plt.title('>> TOP 5 REVENUE DRIVERS - SÃO PAULO', fontsize=14, weight='bold', pad=20)
plt.xlabel('TOTAL REVENUE (BRL)', fontsize=12)
plt.ylabel('PRODUCT CATEGORY', fontsize=12)

# 7. Clean up the edges by removing the top and right borders (despining)
sns.despine(left=True, bottom=True)

# 8. Render the plot!
plt.show()